# 04_1 — ChromaDB Vector Database (Chunk-Level)

Index chunk embeddings in **ChromaDB** running in local persistent mode — no server required.

**Why ChromaDB over raw FAISS:**
- Metadata filtering: `section_type == 'experience'` is applied alongside ANN search
- Structured documents: each entry stores the chunk text + metadata natively
- Persistent by default: survives notebook restarts without re-indexing
- Simple Python API with no external dependencies or running processes

**Inputs** (from notebooks 02 & 03):
- `data/processed/chunk_embeddings.npy` — (N_chunks, 384) float32
- `data/processed/chunks_with_embeddings.json` — chunk metadata aligned by row index
- `data/processed/resume_chunks.json` — full chunk text

**Comparison with 04_2 (FAISS):** ChromaDB stores everything in one place (vectors + metadata + text);
FAISS requires managing separate `.npy` / `.json` files alongside the index.

In [ ]:
# Install chromadb + deps (Python 3.14 needs pydantic-settings for chromadb)
# !pip install chromadb sentence-transformers numpy pydantic-settings

In [2]:
import json
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict
from sentence_transformers import SentenceTransformer
import chromadb

PROJECT_ROOT  = Path("../").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

EMB_PATH    = PROCESSED_DIR / "chunk_embeddings.npy"
META_PATH   = PROCESSED_DIR / "chunks_with_embeddings.json"
CHUNKS_PATH = PROCESSED_DIR / "resume_chunks.json"

CHROMA_DIR  = PROJECT_ROOT / "chroma_store"   # persistent on-disk storage
COLLECTION  = "resume_chunks"
MODEL_NAME  = "all-MiniLM-L6-v2"

print(f"Embeddings  : {EMB_PATH}")
print(f"Chunk meta  : {META_PATH}")
print(f"Chroma store: {CHROMA_DIR}")
print(f"Collection  : {COLLECTION}")

ConfigError: unable to infer type for attribute "chroma_server_nofile"

## 1 — Load chunk embeddings & metadata

In [ ]:
all_embeddings = np.load(EMB_PATH)

with open(META_PATH, "r", encoding="utf-8") as f:
    all_meta = json.load(f)

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

assert len(all_embeddings) == len(all_meta) == len(all_chunks), (
    f"Row count mismatch: embeddings={len(all_embeddings)}, "
    f"meta={len(all_meta)}, chunks={len(all_chunks)}"
)

print(f"Loaded {len(all_embeddings)} chunk embeddings  (dim={all_embeddings.shape[1]})")

section_counts = Counter(m["section_type"] for m in all_meta)
print(f"\n{'Section':<20} {'Chunks':>8}")
print("-" * 30)
for sec, cnt in sorted(section_counts.items(), key=lambda x: -x[1]):
    print(f"{sec:<20} {cnt:>8}")

source_counts = Counter(m["source"] for m in all_meta)
print(f"\n{'Source':<20} {'Chunks':>8}")
print("-" * 30)
for src, cnt in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"{src:<20} {cnt:>8}")

## 2 — Create a persistent ChromaDB client

ChromaDB writes to `chroma_store/` automatically — re-running this cell after
the first index build will load the existing collection instead of re-indexing.
To start fresh, delete the `chroma_store/` directory and re-run.

In [ ]:
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# Persistent client — data survives notebook restarts
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# --- Uncomment for a pure in-memory client (no persistence) ---
# client = chromadb.EphemeralClient()

# get_or_create: safe to re-run; returns existing collection if already built
# We pass embedding_function=None because we supply our own pre-computed vectors
collection = client.get_or_create_collection(
    name=COLLECTION,
    metadata={"hnsw:space": "cosine"},   # cosine similarity
)

print(f"ChromaDB client  : {CHROMA_DIR}")
print(f"Collection       : {COLLECTION}")
print(f"Existing vectors : {collection.count()}")

## 3 — Index chunks

Only DS3 member and board chunks go into the production collection.
Each ChromaDB document stores: the embedding vector, the chunk text as the
`document`, and all metadata fields as the `metadata` dict.

In [ ]:
MEMBER_SOURCES = {"ds3_members", "ds3_board"}

if collection.count() > 0:
    print(f"Collection already has {collection.count()} vectors — skipping re-index.")
    print("Delete chroma_store/ and re-run to rebuild from scratch.")
else:
    ids, embeddings_list, documents, metadatas = [], [], [], []

    for i, (meta, chunk, vec) in enumerate(zip(all_meta, all_chunks, all_embeddings)):
        if meta["source"] not in MEMBER_SOURCES:
            continue

        # ChromaDB metadata values must be str / int / float — no nested dicts or lists
        flat_meta = {
            "chunk_id":     meta["chunk_id"],
            "candidate_id": meta["candidate_id"],
            "source":       meta["source"],
            "section_type": meta["section_type"],
        }
        # Flatten any scalar fields from the chunk-level metadata
        for k, v in (chunk.get("metadata") or {}).items():
            if k in flat_meta:
                continue
            if isinstance(v, (str, int, float, bool)):
                flat_meta[k] = v
            elif isinstance(v, list):
                flat_meta[k] = ", ".join(str(x) for x in v)  # ChromaDB needs scalar

        ids.append(meta["chunk_id"])
        embeddings_list.append(vec.tolist())
        documents.append(chunk["text"])
        metadatas.append(flat_meta)

    # Upsert in batches (ChromaDB default batch limit is 5 461 items)
    BATCH = 512
    for start in range(0, len(ids), BATCH):
        collection.upsert(
            ids=ids[start:start + BATCH],
            embeddings=embeddings_list[start:start + BATCH],
            documents=documents[start:start + BATCH],
            metadatas=metadatas[start:start + BATCH],
        )

    print(f"Indexed {collection.count()} chunks into '{COLLECTION}'")

sec_dist = Counter(m["section_type"] for m in metadatas) if "metadatas" in dir() else {}
if sec_dist:
    for sec, cnt in sorted(sec_dist.items(), key=lambda x: -x[1]):
        print(f"  {sec:<20} {cnt}")

## 4 — Load the embedding model

In [ ]:
model = SentenceTransformer(MODEL_NAME)
print(f"Model loaded: {MODEL_NAME}  (dim={model.get_sentence_embedding_dimension()})")

## 5 — Search functions

Two search modes:
1. **`search_chunks`** — returns individual chunk hits, optional `section_type` / `source` filter
2. **`search_candidates`** — rolls up chunk hits to candidate level (max-score pooling)

ChromaDB's `where` clause supports `$eq`, `$ne`, `$in`, `$and`, `$or` operators.

In [ ]:
def search_chunks(
    query: str,
    top_k: int = 20,
    section_filter: str | None = None,
    source_filter: str | None = None,
):
    """
    Semantic search over ChromaDB chunk vectors.
    Filters use ChromaDB's `where` clause — applied alongside the ANN search.
    ChromaDB returns distances in [0, 2] for cosine space; we convert to
    similarity scores in [0, 1] via  score = 1 - distance / 2.
    """
    q_vec = model.encode([query], normalize_embeddings=True)[0].tolist()

    # Build where clause
    conditions = []
    if section_filter:
        conditions.append({"section_type": {"$eq": section_filter}})
    if source_filter:
        conditions.append({"source": {"$eq": source_filter}})

    where = {"$and": conditions} if len(conditions) > 1 else (conditions[0] if conditions else None)

    kwargs = dict(
        query_embeddings=[q_vec],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    if where:
        kwargs["where"] = where

    results = collection.query(**kwargs)

    hits = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        hits.append({
            "rank":         len(hits) + 1,
            "score":        round(1.0 - dist / 2.0, 4),   # cosine similarity
            "candidate_id": meta["candidate_id"],
            "section_type": meta["section_type"],
            "source":       meta["source"],
            "text":         doc[:300],
        })
    return hits


def search_candidates(
    query: str,
    top_k_candidates: int = 10,
    chunks_per_candidate: int = 3,
    section_filter: str | None = None,
):
    """
    Roll up chunk-level results to candidate-level (max-score pooling).
    """
    raw = search_chunks(
        query,
        top_k=top_k_candidates * chunks_per_candidate * 3,
        section_filter=section_filter,
    )

    candidate_hits: dict = defaultdict(list)
    for hit in raw:
        candidate_hits[hit["candidate_id"]].append(hit)

    candidates = []
    for cid, hits in candidate_hits.items():
        hits_sorted = sorted(hits, key=lambda h: -h["score"])
        candidates.append({
            "candidate_id": cid,
            "best_score":   hits_sorted[0]["score"],
            "top_chunks":   hits_sorted[:chunks_per_candidate],
        })

    candidates.sort(key=lambda c: -c["best_score"])
    return candidates[:top_k_candidates]


print("Search functions ready.")

## 6 — Test: unfiltered chunk search

In [ ]:
test_queries = [
    "Python machine learning data science",
    "React JavaScript full-stack web development",
    "C++ computer vision embedded systems",
]

for query in test_queries:
    print(f"\n{'=' * 70}")
    print(f"Query: {query}")
    print(f"{'=' * 70}")
    results = search_chunks(query, top_k=5)
    for r in results:
        print(f"  #{r['rank']}  score={r['score']:.4f}  [{r['section_type']}]  {r['candidate_id']}")
        print(f"       {r['text'][:120]}")

## 7 — Test: filtered search (section_type)

ChromaDB's `where` clause filters metadata before returning results.
Combined `$and` filters on multiple fields are also supported.

In [ ]:
filtered_queries = [
    ("built APIs with Python FastAPI",          "experience"),
    ("SQL database analytics",                  "skills"),
    ("machine learning neural network project", "projects"),
    ("computer science degree",                 "education"),
]

for query, section in filtered_queries:
    print(f"\n{'=' * 70}")
    print(f"Query [{section} only]: {query}")
    print(f"{'=' * 70}")
    results = search_chunks(query, top_k=3, section_filter=section)
    for r in results:
        print(f"  #{r['rank']}  score={r['score']:.4f}  {r['candidate_id']}")
        print(f"       {r['text'][:150]}")

## 8 — Test: candidate-level rollup

In [ ]:
query = "Python machine learning data science model deployment"
print(f"Query: {query}\n")

candidates = search_candidates(query, top_k_candidates=5)
for c in candidates:
    print(f"  Candidate : {c['candidate_id']}  (best_score={c['best_score']:.4f})")
    for chunk in c["top_chunks"]:
        print(f"    [{chunk['section_type']}] score={chunk['score']:.4f}  {chunk['text'][:100]}")
    print()

## 9 — ChromaDB vs FAISS comparison

Run the same query through both backends and compare results side-by-side.

In [ ]:
import faiss

FAISS_PATH = PROJECT_ROOT / "resume_index.faiss"
FAISS_META = PROJECT_ROOT / "member_chunks_metadata.json"

if FAISS_PATH.exists() and FAISS_META.exists():
    faiss_index = faiss.read_index(str(FAISS_PATH))
    with open(FAISS_META) as f:
        faiss_meta = json.load(f)

    query = "Python machine learning data science"
    q_vec = model.encode([query], normalize_embeddings=True).astype("float32")

    # FAISS search
    scores_f, indices_f = faiss_index.search(q_vec, 5)
    faiss_results = [
        (faiss_meta[i]["candidate_id"], faiss_meta[i]["section_type"], round(float(s), 4))
        for i, s in zip(indices_f[0], scores_f[0]) if i >= 0
    ]

    # ChromaDB search
    chroma_results = [
        (r["candidate_id"], r["section_type"], r["score"])
        for r in search_chunks(query, top_k=5)
    ]

    print(f"Query: {query}\n")
    print(f"{'Rank':<5} {'FAISS':<50} {'ChromaDB':<50}")
    print("-" * 105)
    for i, (f_res, c_res) in enumerate(zip(faiss_results, chroma_results)):
        f_str = f"{f_res[0]}  [{f_res[1]}]  {f_res[2]}"
        c_str = f"{c_res[0]}  [{c_res[1]}]  {c_res[2]}"
        print(f"{i+1:<5} {f_str:<50} {c_str:<50}")
else:
    print("Run 04_2_faiss_indexing.ipynb first to generate the FAISS artifacts.")

## Summary

| Feature | FAISS (04_2) | ChromaDB (04_1) |
|---------|-------------|-----------------|
| Search type | Exact ANN | HNSW (approx.) |
| Metadata filter | Post-hoc in Python | Native `where` clause |
| Metadata storage | External JSON file | Embedded in collection |
| Text storage | External JSON file | Embedded as `document` |
| Persistence | `.faiss` file | `chroma_store/` directory |
| Re-index on restart | Yes (load from file) | No (persistent by default) |
| Best for | Fast exact search, no overhead | Filtered search, self-contained store |

Both feed into the same downstream reranker (`reranker.ipynb`).

**ChromaDB distance note:** ChromaDB returns cosine *distance* in `[0, 2]`.
We convert to similarity with `score = 1 - distance / 2` so scores are
comparable to FAISS inner-product scores (both in `[0, 1]`).